In [1]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)

True

In [2]:
# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [3]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")

In [4]:
# Here is the final output

print(result.final_output)

Autonomous AI agents are just like interns: they’re extremely confident, need very little sleep, and will absolutely do the wrong thing at scale.


In [5]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': 'msg_0c44c338bdd1938c006ad0cd253b4087d290c02e1a7a2ee9df',
  'content': [{'annotations': [],
    'text': 'Autonomous AI agents are just like interns: they’re extremely confident, need very little sleep, and will absolutely do the wrong thing at scale.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [6]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Autonomous AI agents are like interns who never sleep, never ask for coffee, and still somehow send the wrong email to 400 people.


In [7]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Sure — here are 5 AI agent jokes:

1. **Why did the AI agent get promoted?**  
   Because it always followed through on tasks… unless it hallucinated a better plan.

2. **What’s an AI agent’s favorite type of meeting?**  
   A prompt-and-response session.

3. **Why did the AI agent bring a ladder to work?**  
   It wanted to reach the next token.

4. **Why was the AI agent bad at relationships?**  
   It kept saying, “I understand,” without actually understanding.

5. **What do you call an AI agent that loves multitasking?**  
   A promptioneer.

If you want, I can make them **nerdier, darker, or more office-themed**.

In [8]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [9]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [10]:
push("HEY!!")

Push: HEY!!


In [11]:
# Now this (funcion tool object):

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [12]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000020330726750>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [13]:
push_tool.description

'Send the given message to the user as a push notification'

In [14]:
notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [15]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)

Done.


In [16]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [17]:
response = await Runner.run(agent, "Hi there. My name is Altay.")
print(response.final_output)

Hi Altay — nice to meet you! How can I help?


In [18]:
response.to_input_list()

[{'content': 'Hi there. My name is Altay.', 'role': 'user'},
 {'id': 'msg_073a862714202a28006ad0cd2d85fc87d2bc32cfede479bb33',
  'content': [{'annotations': [],
    'text': 'Hi Altay — nice to meet you! How can I help?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [19]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Altay.', 'role': 'user'},
 {'id': 'msg_073a862714202a28006ad0cd2d85fc87d2bc32cfede479bb33',
  'content': [{'annotations': [],
    'text': 'Hi Altay — nice to meet you! How can I help?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name?"}]

In [20]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Altay.
